In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_karkidi_jobs(keyword="data science", pages=1):
    headers = {'User-Agent': 'Mozilla/5.0'}
    base_url = "https://www.karkidi.com/Find-Jobs/{page}/all/India?search={query}"
    jobs_list = []

    for page in range(1, pages + 1):
        url = base_url.format(page=page, query=keyword.replace(' ', '%20'))
        print(f"Scraping page: {page} -> {url}")
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, "html.parser")

        job_blocks = soup.find_all("div", class_="ads-details")
        for job in job_blocks:
            try:
                title = job.find("h4").get_text(strip=True)
                company = job.find("a", href=lambda x: x and "Employer-Profile" in x).get_text(strip=True)
                location = job.find("p").get_text(strip=True)
                experience = job.find("p", class_="emp-exp").get_text(strip=True)
                key_skills_tag = job.find("span", string="Key Skills")
                skills = key_skills_tag.find_next("p").get_text(strip=True) if key_skills_tag else ""
                summary_tag = job.find("span", string="Summary")
                summary = summary_tag.find_next("p").get_text(strip=True) if summary_tag else ""

                jobs_list.append({
                    "Title": title,
                    "Company": company,
                    "Location": location,
                    "Experience": experience,
                    "Summary": summary,
                    "Skills": skills
                })
            except Exception as e:
                print(f"Error parsing job block: {e}")
                continue

        time.sleep(1)

    return pd.DataFrame(jobs_list)

if __name__ == "__main__":
    df_jobs = scrape_karkidi_jobs(keyword="data science", pages=2)
    df_jobs.to_csv("karkidi_jobs.csv", index=False)
    print(df_jobs.head())


Scraping page: 1 -> https://www.karkidi.com/Find-Jobs/1/all/India?search=data%20science
Scraping page: 2 -> https://www.karkidi.com/Find-Jobs/2/all/India?search=data%20science
                                               Title         Company  \
0          Machine Learning Physical Design Engineer          Google   
1  Staff Software Engineer - Monetization, Poe (R...     Quora, Inc.   
2  Staff Backend Engineer - Bot Creator Ecosystem...     Quora, Inc.   
3  Senior Backend Engineer - Bot Creator Ecosyste...     Quora, Inc.   
4                         Data Scientist Lead - AIML  JPMorgan Chase   

                      Location Experience  \
0  Bengaluru, Karnataka, India   4-6 year   
1                        India  8-10 year   
2                        India  8-10 year   
3                        India   6-8 year   
4  Bengaluru, Karnataka, India   6-8 year   

                                             Summary  \
0  Minimum qualifications:Bachelor's degree in El...   
1  About

In [5]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer

# Basic cleanup: lowercase, remove punctuation, split by commas
df_jobs['Cleaned_Skills'] = df_jobs['Skills'].str.lower().str.replace(r'[^a-zA-Z, ]', '', regex=True)

# TF-IDF Vectorization
def custom_tokenizer(x):
    return x.split(',')

vectorizer = TfidfVectorizer(tokenizer=custom_tokenizer, lowercase=True)
X = vectorizer.fit_transform(df_jobs['Cleaned_Skills'])


In [6]:
import joblib
joblib.dump(vectorizer, 'skill_vectorizer.pkl')


['skill_vectorizer.pkl']

In [7]:
from sklearn.cluster import KMeans

k = 5  # You can optimize this using silhouette score or elbow method
kmeans = KMeans(n_clusters=k, random_state=42)
df_jobs['Cluster'] = kmeans.fit_predict(X)

# Save model for future use
import joblib
joblib.dump(kmeans, 'job_cluster_model.pkl')
joblib.dump(vectorizer, 'skill_vectorizer.pkl')


['skill_vectorizer.pkl']

In [8]:
new_jobs = scrape_karkidi_jobs(keyword="data science", pages=1)
new_jobs['Cleaned_Skills'] = new_jobs['Skills'].str.lower().str.replace(r'[^a-zA-Z, ]', '', regex=True)
X_new = vectorizer.transform(new_jobs['Cleaned_Skills'])
new_jobs['Cluster'] = kmeans.predict(X_new)


Scraping page: 1 -> https://www.karkidi.com/Find-Jobs/1/all/India?search=data%20science


In [9]:
user_interest_cluster = [0, 2]  # Example: user prefers clusters 0 and 2
matches = new_jobs[new_jobs['Cluster'].isin(user_interest_cluster)]
if not matches.empty:
    print("🎯 New Matching Jobs Found!")
    print(matches[['Title', 'Company', 'Skills']])


🎯 New Matching Jobs Found!
                                       Title         Company  \
0  Machine Learning Physical Design Engineer          Google   
4                 Data Scientist Lead - AIML  JPMorgan Chase   
5  Applied AI ML Director - Machine Learning  JPMorgan Chase   
6                    Senior Product Designer      Observe.AI   
7                 Manager - Machine Learning      Observe.AI   

                                              Skills  
0  Aartificial intelligence,Algorithms,Data struc...  
4  Aartificial intelligence,Data science techniqu...  
5  Aartificial intelligence,AWS,Azure,Google Clou...  
6  Design,Leadership Skill,Machine learning techn...  
7  Aartificial intelligence,Large Language Models...  


In [11]:
!pip install schedule


In [ ]:
import schedule
import time

def daily_job():
    print("Running job...")  # Replace with your scrape + cluster + alert logic

# Run every day at 09:00
schedule.every().day.at("09:00").do(daily_job)

# Keep script running
while True:
    schedule.run_pending()
    time.sleep(60)
